# Cardiac Segmentation — CAMUS (LV / myocardium / left atrium)

Segmentation, not classification: the clinical need is a *measurement* (chamber size, wall
thickness, ejection fraction), not a category — and CAMUS's ground truth is itself pixel masks
(labels 0=background, 1=LV endocardium, 2=myocardium, 3=left atrium), so the data format dictates
the task the same way it dictated classification for lung/gallbladder in the opposite direction.

Uses the **4CH view, ED + ES frames only** (the frames that actually have ground truth masks —
`half_sequence` files are unlabeled video, not used here). 2CH is a straightforward extension
later using the same pipeline.

**3-candidate bake-off, per the plan**:
1. **"nnU-Net-style" U-Net** — wide (`base_channels=64`), the accuracy-first candidate.
2. **Lightweight U-Net** — narrow (`base_channels=8`, ~2M params per the 2025 IEEE IUS reference),
   the speed-first candidate — reportedly statistically equivalent Dice at 4x faster inference.
3. **UltraSam-init** — flagged as an **optional stretch cell near the end**, not run by default:
   it needs the UltraSam checkpoint downloaded separately, which isn't in hand yet (unlike USCL
   for lung). Skip it unless you've fetched those weights.

Run this notebook **twice** — once with `MODEL_VARIANT = 'standard'`, once with `'lightweight'` —
and compare per-class Dice + inference time to pick the winner, per the bake-off plan.

**Before running**: upload `Cardiac/CAMUS_nifti/` (the 500 `patientXXXX.zip` files) and
`manifests/cardiac_manifest.csv` to your Google Drive under `MyDrive/POCUS-Project/`, preserving
this relative structure.


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install -q nibabel opencv-python-headless


In [ ]:
import zipfile
import tempfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

DRIVE_ROOT = Path('/content/drive/MyDrive/POCUS-Project')
MANIFEST_PATH = DRIVE_ROOT / 'manifests' / 'cardiac_manifest.csv'
ZIP_DIR = DRIVE_ROOT / 'Cardiac' / 'CAMUS_nifti'
CACHE_DIR = Path('/content/cardiac_cache')  # local Colab disk, not Drive -- ephemeral, rebuilt each session
CACHE_DIR.mkdir(exist_ok=True)

VIEW = '4CH'
STRUCTURES = {0: 'background', 1: 'LV', 2: 'myocardium', 3: 'LA'}
NUM_CLASSES = len(STRUCTURES)
IMG_SIZE = 256

MODEL_VARIANT = 'standard'  # 'standard' (base_channels=64) or 'lightweight' (base_channels=8) -- run both, compare
BASE_CHANNELS = 64 if MODEL_VARIANT == 'standard' else 8

BATCH_SIZE = 8
EPOCHS = 30
LR = 1e-3

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, '| Model variant:', MODEL_VARIANT, '| base_channels:', BASE_CHANNELS)


## 2. Cache ED/ES frames + masks locally

Extracting straight from the Drive-hosted zips on every epoch would be slow (500 zips, repeated
nifti decode). Decompress once into local Colab disk as resized `.npy` arrays — this cache is
ephemeral (gone when the runtime resets), which is fine since rebuilding it is a one-time,
few-minute cost per session.


In [ ]:
def load_nifti_from_zip(zf: zipfile.ZipFile, member: str, tmpdir: str) -> np.ndarray:
    path = zf.extract(member, tmpdir)
    return nib.load(path).get_fdata()


def build_cache(manifest_df: pd.DataFrame):
    view_col = f'has_{VIEW}'
    patients = manifest_df[manifest_df[view_col]]['patient_id'].tolist()

    with tempfile.TemporaryDirectory() as tmpdir:
        for patient_id in patients:
            out_path = CACHE_DIR / f'{patient_id}.npz'
            if out_path.exists():
                continue
            zip_path = ZIP_DIR / f'{patient_id}.zip'
            with zipfile.ZipFile(zip_path) as zf:
                arrays = {}
                for phase in ['ED', 'ES']:
                    img = load_nifti_from_zip(zf, f'{patient_id}/{patient_id}_{VIEW}_{phase}.nii.gz', tmpdir)
                    gt = load_nifti_from_zip(zf, f'{patient_id}/{patient_id}_{VIEW}_{phase}_gt.nii.gz', tmpdir)
                    img = cv2.resize(img.astype(np.float32), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
                    gt = cv2.resize(gt.astype(np.uint8), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
                    arrays[f'{phase}_img'] = img / 255.0
                    arrays[f'{phase}_gt'] = gt
                np.savez_compressed(out_path, **arrays)

    print(f'Cached {len(patients)} patients to {CACHE_DIR}')


manifest = pd.read_csv(MANIFEST_PATH)
build_cache(manifest)


## 3. Train/val split — by patient, not by frame

ED and ES frames from the same patient are near-duplicates (same anatomy, same probe position) --
splitting after caching would leak a patient's ED frame into train and their ES frame into val.
Split the 500 patients first.


In [ ]:
view_col = f'has_{VIEW}'
patients_df = manifest[manifest[view_col]]
train_patients, val_patients = train_test_split(patients_df['patient_id'].tolist(), test_size=0.2, random_state=42)
print(f'Train patients: {len(train_patients)}, Val patients: {len(val_patients)}')


## 4. Dataset

Each patient contributes 2 samples (ED + ES frames), both drawn from the same cached `.npz`.

**Augmentation** (train split only): CAMUS's 500 patients is genuinely enough data for this task
(nnU-Net's own published Dice scores of 0.93/0.86/0.89 prove it, on this exact dataset) — unlike
the lung notebook, this isn't a data-scarcity problem. But this pipeline previously had *no*
augmentation at all, while nnU-Net's strong published numbers lean on extensive augmentation +
deep supervision + ensembling. Adding basic augmentation (flip, mild rotation, brightness/contrast
jitter) is a cheap way to close some of that gap without needing more data.

Rotation and flip are applied identically to the image **and** the mask (a spatial transform that
moved the heart but not its outline would just teach the model wrong labels) — nearest-neighbor
interpolation for the mask keeps its values as valid integer labels (0-3), never inventing new
ones the way linear interpolation would. Brightness/contrast jitter only touches the image (a mask
has no "brightness" to jitter).


In [ ]:
def augment(image, mask):
    """Applied to train samples only (see markdown above). Flip/rotation apply identically to
    image and mask so the outline stays aligned with the anatomy; brightness/contrast only
    touches the image."""
    if np.random.rand() < 0.5:
        image = np.fliplr(image).copy()
        mask = np.fliplr(mask).copy()

    angle = np.random.uniform(-15, 15)
    h, w = image.shape
    rot_matrix = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    image = cv2.warpAffine(image, rot_matrix, (w, h), flags=cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    mask = cv2.warpAffine(mask, rot_matrix, (w, h), flags=cv2.INTER_NEAREST,
                           borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    brightness = np.random.uniform(0.9, 1.1)
    contrast = np.random.uniform(0.9, 1.1)
    image = np.clip(image * brightness, 0, 1)
    mean = image.mean()
    image = np.clip((image - mean) * contrast + mean, 0, 1)

    return image.astype(np.float32), mask.astype(np.uint8)


class CamusSegDataset(Dataset):
    def __init__(self, patient_ids: list, train: bool):
        self.train = train
        self.samples = []
        for pid in patient_ids:
            data = np.load(CACHE_DIR / f'{pid}.npz')
            for phase in ['ED', 'ES']:
                self.samples.append((data[f'{phase}_img'], data[f'{phase}_gt']))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img, gt = self.samples[idx]
        if self.train:
            img, gt = augment(img, gt)
        image = torch.from_numpy(img).float().unsqueeze(0)  # 1xHxW
        mask = torch.from_numpy(gt.astype(np.int64))  # HxW, values 0-3
        return image, mask


train_dataset = CamusSegDataset(train_patients, train=True)
val_dataset = CamusSegDataset(val_patients, train=False)
print(f'Train frames: {len(train_dataset)}, Val frames: {len(val_dataset)}')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)


## 5. U-Net

Standard encoder-decoder with skip connections. `base_channels` is the one knob that turns this
into either bake-off candidate — 64 for the accuracy-first "nnU-Net-style" version, 8 for the
lightweight version (~2M params, per the 2025 IEEE IUS reference architecture).


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_ch=1, num_classes=4, base_channels=64):
        super().__init__()
        c = base_channels
        self.enc1 = DoubleConv(in_ch, c)
        self.enc2 = DoubleConv(c, c * 2)
        self.enc3 = DoubleConv(c * 2, c * 4)
        self.enc4 = DoubleConv(c * 4, c * 8)
        self.bottleneck = DoubleConv(c * 8, c * 16)
        self.pool = nn.MaxPool2d(2)

        self.up4 = nn.ConvTranspose2d(c * 16, c * 8, 2, stride=2)
        self.dec4 = DoubleConv(c * 16, c * 8)
        self.up3 = nn.ConvTranspose2d(c * 8, c * 4, 2, stride=2)
        self.dec3 = DoubleConv(c * 8, c * 4)
        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, 2, stride=2)
        self.dec2 = DoubleConv(c * 4, c * 2)
        self.up1 = nn.ConvTranspose2d(c * 2, c, 2, stride=2)
        self.dec1 = DoubleConv(c * 2, c)

        self.out_conv = nn.Conv2d(c, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)


model = UNet(in_ch=1, num_classes=NUM_CLASSES, base_channels=BASE_CHANNELS).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'{MODEL_VARIANT} U-Net: {n_params:,} parameters')


## 6. Loss — CrossEntropy + soft Dice

Cross-entropy alone under-weights small structures (LV/myocardium/LA are a small fraction of
pixels vs. background) — adding a soft Dice term directly optimizes the metric we actually
evaluate on and is standard practice for this kind of class imbalance.


In [ ]:
def soft_dice_loss(logits, targets, num_classes, eps=1e-6):
    probs = F.softmax(logits, dim=1)
    targets_onehot = F.one_hot(targets, num_classes).permute(0, 3, 1, 2).float()
    dims = (0, 2, 3)
    intersection = (probs * targets_onehot).sum(dims)
    union = probs.sum(dims) + targets_onehot.sum(dims)
    dice_per_class = (2 * intersection + eps) / (union + eps)
    return 1 - dice_per_class[1:].mean()  # skip background class 0


ce_loss = nn.CrossEntropyLoss()


def combined_loss(logits, targets):
    return ce_loss(logits, targets) + soft_dice_loss(logits, targets, NUM_CLASSES)


## 7. Train

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def dice_per_class(logits, targets, num_classes, eps=1e-6):
    preds = logits.argmax(dim=1)
    dices = {}
    for c in range(1, num_classes):  # skip background
        pred_c = (preds == c).float()
        target_c = (targets == c).float()
        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum()
        dices[STRUCTURES[c]] = ((2 * intersection + eps) / (union + eps)).item()
    return dices


def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    total_loss = 0.0
    dice_accum = {name: [] for name in list(STRUCTURES.values())[1:]}
    with torch.set_grad_enabled(train):
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            logits = model(images)
            loss = combined_loss(logits, masks)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            for name, val in dice_per_class(logits, masks, NUM_CLASSES).items():
                dice_accum[name].append(val)
    avg_loss = total_loss / len(loader.dataset)
    avg_dice = {name: float(np.mean(vals)) for name, vals in dice_accum.items()}
    return avg_loss, avg_dice


best_val_loss = float('inf')
for epoch in range(EPOCHS):
    train_loss, _ = run_epoch(train_loader, train=True)
    val_loss, val_dice = run_epoch(val_loader, train=False)
    dice_str = ', '.join(f'{k}={v:.3f}' for k, v in val_dice.items())
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} | {dice_str}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), DRIVE_ROOT / 'Cardiac' / f'cardiac_unet_{MODEL_VARIANT}_best.pth')

print('\nBest val loss:', best_val_loss)


## 8. Final evaluation — Dice + inference speed

Report both, since the bake-off criterion is Dice *weighed against* inference speed (the
lightweight variant's whole pitch is trading a little accuracy for ~4x faster inference).


In [ ]:
import time

model.load_state_dict(torch.load(DRIVE_ROOT / 'Cardiac' / f'cardiac_unet_{MODEL_VARIANT}_best.pth'))
model.eval()

_, final_dice = run_epoch(val_loader, train=False)
print(f'[{MODEL_VARIANT}] Final per-class Dice:', final_dice)
print(f'[{MODEL_VARIANT}] Benchmark target (nnU-Net on CAMUS): LV=0.93, myocardium=0.86, LA=0.89')

# Inference speed on a single frame, averaged
sample_image, _ = val_dataset[0]
sample_image = sample_image.unsqueeze(0).to(device)
with torch.no_grad():
    for _ in range(5):  # warmup
        model(sample_image)
    start = time.time()
    n_runs = 50
    for _ in range(n_runs):
        model(sample_image)
    elapsed_ms = (time.time() - start) / n_runs * 1000

print(f'[{MODEL_VARIANT}] Inference time: {elapsed_ms:.2f} ms/frame ({n_params:,} params)')


## 9. [Optional / stretch] UltraSam-init as a 3rd candidate

**Not run by default** — requires the UltraSam checkpoint downloaded separately (not yet in hand,
unlike USCL for lung). Only run this if you've fetched those weights; otherwise the standard vs.
lightweight comparison above is the real bake-off for now.


In [ ]:
USE_ULTRASAM = False  # flip to True once you've downloaded UltraSam weights
ULTRASAM_CKPT = DRIVE_ROOT / 'Cardiac' / 'ultrasam_weights' / 'ultrasam.pth'

if USE_ULTRASAM and ULTRASAM_CKPT.exists():
    print('UltraSam checkpoint found -- wire up its encoder as the U-Net backbone here.')
    # Loading recipe depends on UltraSam's released checkpoint format -- inspect its state_dict
    # keys the same way we did for USCL before writing the loading code.
else:
    print('Skipping UltraSam candidate: weights not present. Set USE_ULTRASAM=True once fetched.')


## 10. External validation — archive.zip (different-source LV masks)

A second, independently-sourced dataset (900 frame/mask PNG pairs, single binary LV class, no
patient/clip metadata, unknown provenance). Used purely as a **held-out generalization check** —
deliberately **not** folded into training:

- Only LV is labeled here (vs. CAMUS's 3-class LV/myocardium/LA masks), so it can't supervise the
  other two classes without a partial-label loss rework.
- No patient/clip grouping is available, so an internal train/val split of this set risks leaking
  near-duplicate frames across the split with no way to detect it.

Neither problem matters for pure evaluation: keep all 900 frames held out, run the CAMUS-trained
model's LV channel against them, and report Dice. This tests whether the model generalizes past
CAMUS's specific acquisition style, or is overfit to it.

**Before running**: upload `Cardiac/archive.zip` to Drive under `MyDrive/POCUS-Project/Cardiac/`.


In [ ]:
import io
from PIL import Image

ARCHIVE_ZIP_PATH = DRIVE_ROOT / 'Cardiac' / 'archive.zip'


def load_archive_frames_masks(zip_path, img_size):
    """Loads archive.zip's frame_XXXX.png / mask_XXXX.png pairs, resized/normalized the same way
    as the CAMUS cache (see build_cache above) so the trained model sees consistent input stats.
    Masks are binarized -- this dataset only distinguishes LV (foreground) vs background."""
    frames, masks = [], []
    with zipfile.ZipFile(zip_path) as zf:
        names = set(zf.namelist())
        frame_names = sorted(n for n in names if n.startswith('frames/frame_') and n.endswith('.png'))
        for frame_name in frame_names:
            idx = frame_name.split('frame_')[1].split('.')[0]
            mask_name = f'masks/mask_{idx}.png'
            if mask_name not in names:
                continue
            img = np.array(Image.open(io.BytesIO(zf.read(frame_name))).convert('L'))
            mask = np.array(Image.open(io.BytesIO(zf.read(mask_name))).convert('L'))
            img = cv2.resize(img.astype(np.float32), (img_size, img_size), interpolation=cv2.INTER_LINEAR)
            mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
            frames.append(img / 255.0)
            masks.append((mask > 127).astype(np.uint8))  # binarize -- foreground = LV
    return np.stack(frames), np.stack(masks)


archive_imgs, archive_masks = load_archive_frames_masks(ARCHIVE_ZIP_PATH, IMG_SIZE)
print(f'Loaded {len(archive_imgs)} frame/mask pairs from archive.zip')


class ArchiveValidationDataset(Dataset):
    def __init__(self, imgs, masks):
        self.imgs = imgs
        self.masks = masks

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        image = torch.from_numpy(self.imgs[idx]).float().unsqueeze(0)
        mask = torch.from_numpy(self.masks[idx]).long()
        return image, mask


archive_dataset = ArchiveValidationDataset(archive_imgs, archive_masks)
archive_loader = DataLoader(archive_dataset, batch_size=BATCH_SIZE)


In [ ]:
model.load_state_dict(torch.load(DRIVE_ROOT / 'Cardiac' / f'cardiac_unet_{MODEL_VARIANT}_best.pth'))
model.eval()


def lv_dice_against_binary(loader, eps=1e-6):
    """Per-frame LV Dice: model prediction class 1 (LV) vs. this dataset's binary foreground mask."""
    dices = []
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            preds = model(images).argmax(dim=1)
            pred_lv = (preds == 1).float()
            true_lv = masks.float()
            for i in range(images.size(0)):
                inter = (pred_lv[i] * true_lv[i]).sum()
                union = pred_lv[i].sum() + true_lv[i].sum()
                dices.append(((2 * inter + eps) / (union + eps)).item())
    return dices


archive_dices = lv_dice_against_binary(archive_loader)
print(f'[{MODEL_VARIANT}] archive.zip external validation -- LV Dice: '
      f'mean={np.mean(archive_dices):.3f}, median={np.median(archive_dices):.3f}, '
      f'std={np.std(archive_dices):.3f} (n={len(archive_dices)})')
print(f'Compare against this same model\'s CAMUS val-set LV Dice from section 8 -- a substantially')
print(f'lower number here (not just "not 0.93") is the honest generalization gap, not a bug.')

# Visual sanity check: a few examples, since Dice alone can hide *how* a model is wrong.
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
sample_idxs = np.random.RandomState(42).choice(len(archive_dataset), 4, replace=False)
with torch.no_grad():
    for col, idx in enumerate(sample_idxs):
        image, mask = archive_dataset[idx]
        pred = (model(image.unsqueeze(0).to(device)).argmax(dim=1) == 1).float().cpu().squeeze(0).numpy()
        axes[0, col].imshow(image.squeeze(0).numpy(), cmap='gray')
        axes[0, col].set_title(f'Frame {idx}'); axes[0, col].axis('off')
        axes[1, col].imshow(mask.numpy(), cmap='gray')
        axes[1, col].set_title('Ground truth LV'); axes[1, col].axis('off')
        axes[2, col].imshow(pred, cmap='gray')
        axes[2, col].set_title(f'Predicted (Dice={archive_dices[idx]:.2f})'); axes[2, col].axis('off')
plt.tight_layout()
plt.show()


## 11. Ejection fraction — from segmentation to "impaired ventricular function"

The original spec's cardiac list wants "impaired ventricular function," not just chamber outlines.
CAMUS's own manifest already carries each patient's real clinical ejection fraction (the
`{VIEW}_ejection_fraction` column, from `cardiac_manifest.py`) — unused until now.

This computes a simple **area-based EF proxy** from the model's *predicted* segmentation instead
of the ground-truth mask:

    AEF = (LV_area_ED - LV_area_ES) / LV_area_ED * 100

This is a simplification of real volumetric EF (which needs Simpson's-method disk summation across
a full biplane acquisition) — with only one 2D view here, area is a proxy, not a clinical-grade
measurement. But it tests something real and directly useful: does the model's *own* segmentation
output carry enough signal to track real EF, or does it just look accurate without being
functionally useful for the actual clinical question being asked.


In [ ]:
model.load_state_dict(torch.load(DRIVE_ROOT / 'Cardiac' / f'cardiac_unet_{MODEL_VARIANT}_best.pth'))
model.eval()

ef_col = f'{VIEW}_ejection_fraction'
manifest_by_pid = manifest.set_index('patient_id')

val_ef_true, val_ef_pred = [], []
with torch.no_grad():
    for pid in val_patients:
        true_ef = manifest_by_pid.loc[pid, ef_col]
        if pd.isna(true_ef):
            continue
        data = np.load(CACHE_DIR / f'{pid}.npz')
        areas = {}
        for phase in ['ED', 'ES']:
            image = torch.from_numpy(data[f'{phase}_img']).float().unsqueeze(0).unsqueeze(0).to(device)
            pred = model(image).argmax(dim=1).squeeze(0).cpu().numpy()
            areas[phase] = (pred == 1).sum()  # LV pixel count
        if areas['ED'] == 0:
            continue  # avoid divide-by-zero if the model predicted no LV at all for this frame
        pred_ef = (areas['ED'] - areas['ES']) / areas['ED'] * 100
        val_ef_true.append(float(true_ef))
        val_ef_pred.append(pred_ef)

val_ef_true = np.array(val_ef_true)
val_ef_pred = np.array(val_ef_pred)
mae = np.mean(np.abs(val_ef_true - val_ef_pred))
corr = np.corrcoef(val_ef_true, val_ef_pred)[0, 1]
print(f'[{MODEL_VARIANT}] Area-based EF proxy vs. real clinical EF (n={len(val_ef_true)} val patients):')
print(f'  MAE = {mae:.1f} percentage points, correlation = {corr:.3f}')

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(val_ef_true, val_ef_pred, alpha=0.6)
lims = [min(val_ef_true.min(), val_ef_pred.min()), max(val_ef_true.max(), val_ef_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1, label='perfect agreement')
ax.set_xlabel('Real clinical EF (%)')
ax.set_ylabel('Area-based EF proxy from predicted mask (%)')
ax.set_title(f'{MODEL_VARIANT} — EF agreement (MAE={mae:.1f}pp, r={corr:.2f})')
ax.legend()
plt.tight_layout()
plt.show()

print('\nClinical EF categories, for reference: normal >=55%, mildly reduced 41-54%, moderately')
print('reduced 30-40%, severely reduced <30%. A sanity check beyond raw correlation: does the')
print('proxy at least land in the right *category* most of the time, since that\'s closer to how')
print('this would actually be used clinically than the exact percentage.')


def ef_category(ef):
    if ef >= 55:
        return 'normal'
    if ef >= 41:
        return 'mildly reduced'
    if ef >= 30:
        return 'moderately reduced'
    return 'severely reduced'


true_cats = [ef_category(e) for e in val_ef_true]
pred_cats = [ef_category(e) for e in val_ef_pred]
cat_agreement = np.mean([t == p for t, p in zip(true_cats, pred_cats)])
print(f'Category agreement: {cat_agreement:.1%}')
